# Highway Networks — Paper Exploration

**Goal:** Understand Highway Networks and prepare to implement a classifier for CIFAR-10.

### Notebook outline
1. Setup and environment
2. CIFAR-10 dataset
3. Highway Networks: theory
4. Gate-bias experiment
5. First Highway Layer implementation
6. Testing and gradient flow

> This notebook is for learning and experiments. Once the layer is tested, we can move reusable code to `src/highway.py`.

## 1. Setup and environment

Import the libraries we need and confirm which device PyTorch will use.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("PyTorch:", torch.__version__)

## 2. CIFAR-10 dataset

### 2.1 Dataset overview

- 60,000 color images
- Image dimensions: 32 × 32 pixels
- 50,000 training images
- 10,000 test images
- 10 equally sized classes, with 6,000 images per class overall

Each image has three color channels: red, green, and blue.


Our network will take an image tensor with shape `(3, 32, 32)` and produce ten class scores.

The `3` represents RGB channels. A batch of images will have shape `(batch_size, 3, 32, 32)`.

### 2.2 Load the dataset

`ToTensor()` converts an image into a PyTorch tensor and scales pixel values to `[0, 1]`. The download will happen the first time you run this cell.

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="../data", train=False, download=True, transform=transform
)

classes = train_dataset.classes
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", classes)

### 2.3 Inspect an example image

Each dataset item contains an image tensor and its numeric class label.

In [ ]:
image, label = train_dataset[0]
print("Image tensor shape:", image.shape)
print("Image tensor dtype:", image.dtype)
print("Pixel range:", float(image.min()), "to", float(image.max()))
print("Label:", label, "—", classes[label])

plt.figure(figsize=(3, 3))
plt.imshow(image.permute(1, 2, 0))
plt.title(classes[label])
plt.axis("off")
plt.show()

### 2.4 Visualize sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Highway Networks — Theory

### 3.1 Highway layer equation

$$
y = H(x)T(x) + x(1-T(x))
$$

- $x$: original input
- $H(x)$: transformed input
- $T(x)$: transform gate, controlling the transformed contribution
- $1-T(x)$: carry gate, controlling the original-input contribution

Multiplication is **element-wise**; the input and transformed features must have matching shapes.

### 3.2 How the gates are learned

$$
H(x) = \operatorname{ReLU}(W_Hx+b_H)
$$

$$
T(x) = \sigma(W_Tx+b_T)
$$

$$
\sigma(z) = \frac{1}{1+e^{-z}}
$$

Weights and biases in the transform and gate are **learned during training**. Sigmoid values lie between 0 and 1.

| Transform gate $T(x)$ | Carry gate $1-T(x)$ | Behavior |
|---:|---:|---|
| 0.0 | 1.0 | Carry the input |
| 0.2 | 0.8 | Mostly carry |
| 0.5 | 0.5 | Mix both |
| 0.8 | 0.2 | Mostly transform |
| 1.0 | 0.0 | Transform the input |

### 3.3 Why use a negative gate bias?

If $W_Tx = 0$ and $b_T=-2$, then

$$
T(x)=\sigma(-2)\approx0.119,\qquad 1-T(x)\approx0.881.
$$

Thus the layer initially favors carrying information forward, making it **closer to an identity mapping** ($y\approx x$) in this illustrative case. This helps provide a more direct path for information and gradients in deep networks. Actual gate values also depend on $W_Tx$.

## 4. Gate-bias experiment

Observe how a negative gate bias favors the carry route when the weighted input is zero.

In [ ]:
biases = torch.tensor([-5.0, -2.0, 0.0, 2.0, 5.0])
transform_gate = torch.sigmoid(biases)
carry_gate = 1 - transform_gate

for bias, t, c in zip(biases, transform_gate, carry_gate):
    print(
        f"bias = {bias.item():+4.1f} | "
        f"transform = {t.item():.3f} | "
        f"carry = {c.item():.3f}"
    )

## 5. Implement a fully connected Highway Layer

We start with `nn.Linear` to understand the gating mechanism. This is a learning exercise; we will later adapt the layer for image tensors.

In [ ]:
import torch.nn as nn

class HighwayLayer(nn.Module):
    def __init__(self, features, gate_bias=-2.0):
        super().__init__()
        self.transform = nn.Linear(features, features)
        self.gate = nn.Linear(features, features)
        nn.init.constant_(self.gate.bias, gate_bias)

    def forward(self, x):
        H = torch.relu(self.transform(x))
        T = torch.sigmoid(self.gate(x))
        return H * T + x * (1 - T)

## 6. Test the Highway Layer

### 6.1 Verify input and output shapes

In [ ]:
torch.manual_seed(42)
layer = HighwayLayer(features=4)

x = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0]
])
y = layer(x)

print("Input shape:", x.shape)
print("Output shape:", y.shape)
print("Input:\n", x)
print("Output:\n", y)
assert y.shape == x.shape

### 6.2 Inspect transform and carry gates

In [ ]:
with torch.no_grad():
    H = torch.relu(layer.transform(x))
    T = torch.sigmoid(layer.gate(x))
    C_gate = 1 - T

print("Transform gate T(x):\n", T)
print("Carry gate 1-T(x):\n", C_gate)
print("Sum of gates:\n", T + C_gate)
assert torch.allclose(T + C_gate, torch.ones_like(T))

### 6.3 Confirm gradient flow

Both sets of learnable parameters should receive gradients during backpropagation.

In [ ]:
layer.zero_grad()
x_grad = torch.randn(2, 4, requires_grad=True)
y_grad = layer(x_grad)
loss = y_grad.sum()
loss.backward()

print("Input gradient:\n", x_grad.grad)
print("Transform weight gradient exists:", layer.transform.weight.grad is not None)
print("Gate weight gradient exists:", layer.gate.weight.grad is not None)

assert x_grad.grad is not None
assert layer.transform.weight.grad is not None
assert layer.gate.weight.grad is not None

## 7. Next steps

- [ ] Understand each part of the Highway Layer implementation
- [ ] Move reusable layer code into `src/highway.py`
- [ ] Build a convolutional Highway Layer for image feature maps
- [ ] Assemble and train a CIFAR-10 classifier
- [ ] Evaluate test accuracy and compare with a baseline